In [46]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf


data=pd.read_csv('WAGE2_2025_abridged.csv')

state=530088808+540338216+540527739+530800484+530356855 

data = pd.read_csv('WAGE2_2025_abridged.csv')
cont = ['educ', 'KWW', 'age', 'IQ', 'hours']
for v in ['married', 'urban']:
    data[f'{v}_1'] = (data[v] == 1).astype(int)
def forward_selected(df, response):
    remaining = set(df.columns) - {response}
    selected = []
    best_adj = 0.0
    while remaining:
        candidates = []
        for c in remaining:
            formula = f"{response} ~ {' + '.join(selected + [c])}"
            adj = smf.ols(formula, df).fit().rsquared_adj
            candidates.append((adj, c))
        adj_new, var_new = max(candidates)
        if adj_new > best_adj:
            best_adj, best_var = adj_new, var_new
            selected.append(best_var)
            remaining.remove(best_var)
        else:
            break
    final_formula = f"{response} ~ {' + '.join(selected)}"
    return smf.ols(final_formula, df).fit()

# 3.

## 3.1 M1: log-linear (log(wage) ~ linear predictors)
data['lwage'] = np.log(data['wage'])
vars_M1 = ['lwage'] + cont + [f'{v}_1' for v in ['married','urban']]
M1 = forward_selected(data[vars_M1].dropna(), 'lwage')
print("=== M1: log-linear – FS ===")
print(M1.summary())

## 3.2 M2: linear-log (wage ~ log(predictors))
for c in cont:
    data[f'log_{c}'] = np.log(data[c] + 1e-6)
vars_M2 = ['wage'] + [f'log_{c}' for c in cont] + [f'{v}_1' for v in ['married','urban']]
M2 = forward_selected(data[vars_M2].dropna(), 'wage')
print("\n=== M2: linear-log – FS ===")
print(M2.summary())

## 3.3 M3: log-log (log(wage) ~ log(predictors))
vars_M3 = ['lwage'] + [f'log_{c}' for c in cont] + [f'{v}_1' for v in ['married','urban']]
M3 = forward_selected(data[vars_M3].dropna(), 'lwage')
print("\n=== M3: log-log – FS ===")
print(M3.summary())

## 3.4 M4: spline-linear on hours (wage ~ hours splines + other linear)
knots = data['hours'].quantile([0.2,0.4,0.6,0.8]).values
for i, k in enumerate(knots, 1):
    data[f'Step{i}'] = np.where(data['hours'] > k, data['hours'] - k, 0)
vars_M4 = ['wage', 'hours'] + [f'Step{i}' for i in range(1,5)] + ['educ','KWW','age','IQ','married_1','urban_1']
M4 = forward_selected(data[vars_M4].dropna(), 'wage')
print("\n=== M4: spline-linear – FS ===")
print(M4.summary())

ser_M1 = np.sqrt(M1.mse_resid)
ser_M2 = np.sqrt(M2.mse_resid)
ser_M3 = np.sqrt(M3.mse_resid)
ser_M4 = np.sqrt(M4.mse_resid)

print(f"\nSERs:")
print(f" M1 (log-linear)     = {ser_M1:.3f}")
print(f" M2 (linear-log)     = {ser_M2:.3f}")
print(f" M3 (log-log)        = {ser_M3:.3f}")
print(f" M4 (spline-linear)  = {ser_M4:.3f}")

=== M1: log-linear – FS ===
                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.228
Model:                            OLS   Adj. R-squared:                  0.222
Method:                 Least Squares   F-statistic:                     38.22
Date:                Wed, 28 May 2025   Prob (F-statistic):           4.05e-47
Time:                        00:12:31   Log-Likelihood:                -393.20
No. Observations:                 916   AIC:                             802.4
Df Residuals:                     908   BIC:                             841.0
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      4.9608   

Discussion: Across four finalist models—log‐linear (log wage on linear predictors), linear‐log (wage on log predictors), log‐log (log wage on log predictors), and spline‐linear (wage on hours splines plus linear controls)—we observe that none achieves what we’d call a “good fit” by the conventional benchmark (adjusted R² ≥ 0.50), yet each offers distinct merits. The log-linear and log-log specifications both deliver the highest explanatory power (Adj. R² ≈ 0.256) and the lowest retransformed SER (≈ 0.371), with the log-log model additionally yielding direct elasticity estimates that link percentage changes in education, hours, IQ, and other covariates to percentage changes in wages. The linear-log model, by contrast, eases nonlinearity in inputs like hours and age via log transformations, but suffers from a higher raw-scale SER (~370) and slightly lower adjusted SER. R² (≈ 0.242), making it a secondary choice. Meanwhile, the spline‐linear approach—introducing four evenly spaced knots in hours—achieves the lowest SER on the original scale (≈ 355) by flexibly capturing kinked effects (e.g., overtime thresholds), albeit at the cost of higher complexity (11 predictors) and less intuitive coefficient interpretation. In practice, we therefore recommend the log‐log FS model as the primary workhorse, since it balances parsimonious structure, variance stabilisation, decent fit, and elasticity interpretation; the spline‐linear FS model can serve as a specialised alternative when modelling nonlinear hours effects is paramount. To further enhance predictive performance, one might incorporate additional controls (industry or region), employ penalised regression to guard against overfitting, or validate model choices via cross‐validation.